# Week 11 · Day 3 — Base-Model Baseline & the Grading Harness

**Goal today:** measure how well the *untouched* Qwen2.5-7B model scores on your 100-question exam. This is your **"before" number**. On Day 4 you fine-tune, and on the weekend you compare "after" vs "before" — so today's number must be measured **fairly and repeatably**.

> **Turn the GPU ON today.** This is the first day we load the real 7B model.
> On Kaggle: **Settings → Accelerator → GPU (T4 ×2 or P100)** and **Settings → Internet → ON** (needed to download the model, one time).

## Where we are
- **Day 1 (done):** setup + data inspection (638 rows → only 88 unique).
- **Day 2 (done):** added public datasets (~3,200 unique), cleaned, deduped, made a leakage-safe train/val split, and started the 100-question benchmark.
- **Day 3 (today):** build the grading harness and score the **base model** on the benchmark.

## How to run this on Kaggle
1. **+ Add Input** (right sidebar) → attach the dataset that holds your `rf_mcq_100.jsonl` (or the `rf_mcq_starter.jsonl` from Day 2).
2. Turn **GPU + Internet ON** (sidebar).
3. Run the cells top to bottom. The first model load downloads ~5 GB (cached afterwards).
4. Click **Save Version** at the end to keep the result files.

## The 4 ideas you'll learn today

- **Deterministic decoding** — for a *fair* exam, the model must answer the **same way every time**. We use greedy decoding (`temperature = 0`) and a fixed random seed. No randomness = repeatable scores you can trust and compare later.
- **Log-probability scoring (our main method)** — instead of asking the model to *write* an answer and hoping it says "B", we look **inside** the model at how much probability it gives each letter A/B/C/D as the next token, and pick the highest. It can't be fooled by formatting ("The answer is B." vs "**B**").
- **Answer extraction (the cross-check)** — we *also* let the model generate a few tokens and read the first letter with a simple pattern. Where the two methods **agree**, we're confident; where they disagree, the question is worth a look.
- **Confidence interval** — with only **100 questions**, a score has wiggle room. Rule of thumb: the 95% margin is about **±10 points** near 50%. So a "+10 point" gain later is right at the edge of noise — which is why on Saturday we'll also use a **paired test (McNemar)** to check the improvement is real.

**Why two scoring methods?** Log-prob is the reliable judge; generation is how a human would test it. Reporting both (and their agreement) is what makes this a *serious* evaluation, not a demo.

In [ ]:
# ---- Day 3 needs a GPU. On Kaggle: Settings -> Accelerator -> GPU (T4 x2 or P100),
#      and Settings -> Internet -> ON (to download the model). Then run this cell. ----
# We update only bitsandbytes + accelerate (4-bit loading); Kaggle's transformers is fine.
!pip install -q -U bitsandbytes accelerate

import torch, transformers
print("transformers:", transformers.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
    print("bf16 supported:", torch.cuda.is_bf16_supported(),
          "(T4/P100 = False -> we use fp16, which is correct here)")
else:
    print("NO GPU DETECTED! Turn it on: Settings -> Accelerator -> GPU, then re-run this cell.")

## Step 1 — Load your 100-question benchmark

**What:** read the exam you started on Day 2 (`rf_mcq_100.jsonl`, or `rf_mcq_starter.jsonl` if you haven't finished all 100 yet).

**Why:** this held-out exam is the *only* fair yardstick. The model never trained on it, so the score reflects real ability, not memorization.

**What could go wrong:**
- *File not found* → attach the right dataset with **+ Add Input**, or you're running before Day 2's output was saved. The code searches several locations and, as a last resort, uses a tiny 3-question demo so the notebook still runs — but **attach your real 100** for a meaningful score.
- *Malformed question* → the check prints any question whose options aren't exactly A/B/C/D or whose answer isn't a valid letter.

**How we check:** it prints how many questions loaded, any malformed ids, and the per-domain counts.

In [ ]:
import json, glob, os, re, math, random
from collections import Counter, defaultdict
import torch

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)
LETTERS = ["A", "B", "C", "D"]

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows

# Prefer the finished 100; fall back to the Day-2 starter; search Kaggle + local paths.
candidates = (glob.glob("/kaggle/input/**/rf_mcq_100.jsonl",    recursive=True) +
              glob.glob("/kaggle/working/**/rf_mcq_100.jsonl",  recursive=True) +
              glob.glob("/kaggle/input/**/rf_mcq_starter.jsonl", recursive=True) +
              glob.glob("/kaggle/working/**/rf_mcq_starter.jsonl", recursive=True) +
              glob.glob("data/**/rf_mcq_*.jsonl", recursive=True))

BENCH = []
for p in candidates:
    BENCH = read_jsonl(p)
    if BENCH:
        print("Loaded benchmark:", p, f"({len(BENCH)} questions)")
        break

if not BENCH:
    print("No benchmark file found — using a tiny built-in demo set (ATTACH your rf_mcq_100.jsonl!).")
    BENCH = [
        {"id": "mcq-001", "domain": "RF Fundamentals", "difficulty": "easy",
         "question": "If a signal's power doubles, by how many dB does it increase?",
         "options": {"A": "2 dB", "B": "3 dB", "C": "6 dB", "D": "10 dB"}, "answer": "B"},
        {"id": "mcq-002", "domain": "DSP", "difficulty": "easy",
         "question": "A 30 kHz tone is sampled at 48 kHz. What alias frequency appears?",
         "options": {"A": "6 kHz", "B": "12 kHz", "C": "18 kHz", "D": "No aliasing"}, "answer": "C"},
        {"id": "mcq-003", "domain": "Modulation", "difficulty": "medium",
         "question": "How many bits does each symbol carry in 64-QAM?",
         "options": {"A": "4", "B": "6", "C": "8", "D": "16"}, "answer": "B"},
    ]

# Sanity check: options must be exactly A/B/C/D and answer must be a valid letter.
bad = [q.get("id") for q in BENCH
       if set(q.get("options", {})) != set(LETTERS) or q.get("answer") not in LETTERS]
print("Malformed questions:", bad or "none")
print("Total questions   :", len(BENCH))
print("By domain         :", dict(Counter(q.get("domain", "?") for q in BENCH)))

## Step 2 — Load the base model in 4-bit

**What:** load `Qwen2.5-7B-Instruct` compressed to **4-bit** so it fits in the GPU's 16 GB.

**Why:** this is the exact "professor" we'll fine-tune later. We must score it **before** any changes to get the honest "before" number.

**In plain terms:**
- **4-bit NF4** = the pocket-summary compression (little quality loss).
- **compute dtype = fp16** because Kaggle's T4/P100 GPUs don't support bf16. (Newer GPUs would use bf16.)
- The first load **downloads ~5 GB** — that's why Internet must be ON. It's cached for the rest of the session.

**What could go wrong:**
- *Out of memory / no GPU* → the accelerator isn't on. Turn it on and re-run.
- *Download errors* → Internet is off or a network hiccup; just re-run the cell (it resumes from cache).

**How we check:** it prints that the model loaded and which device it's on.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit NF4 quantization (the "Q" in QLoRA). compute dtype = fp16 (T4/P100 have no bf16).
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()  # inference mode: no gradients, deterministic
print("Base model loaded in 4-bit. First layer on:", next(model.parameters()).device)

## Step 3 — Build the exam prompt and the two scorers

**What:** turn each question into a chat prompt (system message + the question with its A–D options), then define **two** ways to grade it:
1. `score_logprob` — the **primary** judge: compares the model's probability of "A" vs "B" vs "C" vs "D" as the next token and picks the highest. One forward pass, fully deterministic.
2. `score_generate` — the **cross-check**: the model writes a few tokens, and we read the first A/B/C/D.

**Why this design:** the *same* prompt is used for every question (and later for the fine-tuned model), so comparisons are apples-to-apples. Log-prob avoids parsing failures; generation mirrors real use.

**What could go wrong:**
- *A letter maps to an unusual token* → we handle both `"A"` and `" A"` variants when finding each letter's token id.
- *The model rambles when generating* → fine; we only read the first letter, and log-prob is the score of record.

**How we check:** a quick smoke test prints both methods' pick for question 1 next to the correct answer.

In [ ]:
EVAL_SYSTEM = ("You are an expert RF, DSP, and wireless communications engineer taking a "
               "multiple-choice exam. Read the question and choose the single best option.")

def build_messages(q):
    o = q["options"]
    user = (f"{q['question']}\n\n"
            f"A) {o['A']}\nB) {o['B']}\nC) {o['C']}\nD) {o['D']}\n\n"
            "Answer with a single letter: A, B, C, or D.")
    return [{"role": "system", "content": EVAL_SYSTEM},
            {"role": "user",   "content": user}]

# The token id(s) that stand for each answer letter (try bare "A" and space " A").
def _letter_ids():
    ids = {}
    for L in LETTERS:
        cand = set()
        for s in (L, " " + L):
            t = tok(s, add_special_tokens=False).input_ids
            if t:
                cand.add(t[0])
        ids[L] = list(cand)
    return ids
LETTER_IDS = _letter_ids()

def _encode(q):
    # return_dict=True gives {input_ids, attention_mask}; robust across transformers versions.
    enc = tok.apply_chat_template(build_messages(q), add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    return {k: v.to("cuda") for k, v in enc.items()}

@torch.no_grad()
def score_logprob(q):
    """Deterministic: compare the model's probability of A/B/C/D as the next token."""
    enc = _encode(q)
    logits = model(**enc).logits[0, -1].float()        # distribution over the NEXT token
    logp = torch.log_softmax(logits, dim=-1)
    scores = {L: max(logp[i].item() for i in ids) for L, ids in LETTER_IDS.items()}
    return max(scores, key=scores.get), scores

@torch.no_grad()
def score_generate(q):
    """Cross-check: let the model write an answer, then read the first letter."""
    enc = _encode(q)
    out = model.generate(**enc, max_new_tokens=5, do_sample=False,
                         pad_token_id=tok.eos_token_id)   # greedy = deterministic
    text = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"[ABCD]", text.upper())
    return (m.group(0) if m else None), text.strip()

# smoke test on the first question
p_lp, _ = score_logprob(BENCH[0])
p_gen, gen_txt = score_generate(BENCH[0])
print("Q1 correct answer:", BENCH[0]["answer"])
print("Q1 log-prob pick :", p_lp)
print("Q1 generate pick :", p_gen, f"(model wrote: {gen_txt!r})")

## Step 4 — Grade the whole exam (deterministically)

**What:** loop over every question, score it with **both** methods, and save a **per-question** results table.

**Why:** the per-question record is what powers Saturday's deep analysis — which questions the fine-tune *fixed*, which it *broke*, and which both models miss. Saving it now makes today's work reusable.

**What could go wrong:**
- *Very slow* → each question is one small forward pass + a 5-token generation; 100 questions take well under a minute on a T4.

**How we check:** it prints progress and saves `baseline_per_question.csv` to `/kaggle/working/`.

In [ ]:
import csv
from time import time

results = []
t0 = time()
for i, q in enumerate(BENCH, 1):
    lp_pred, _ = score_logprob(q)
    gen_pred, gen_text = score_generate(q)
    gold = q["answer"]
    results.append({
        "id": q.get("id", f"q{i}"),
        "domain": q.get("domain", "?"),
        "difficulty": q.get("difficulty", "?"),
        "gold": gold,
        "pred_logprob": lp_pred,
        "pred_gen": gen_pred,
        "correct": int(lp_pred == gold),        # log-prob is our primary score
        "methods_agree": int(lp_pred == gen_pred),
        "gen_text": gen_text,
    })
    if i % 20 == 0:
        print(f"  scored {i}/{len(BENCH)} ...")
print(f"Done grading in {time() - t0:.1f}s")

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/baseline_per_question.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    w.writeheader()
    w.writerows(results)
print("Saved:", f"{OUT}/baseline_per_question.csv")

## Step 5 — Read the baseline scores

**What:** compute the **overall accuracy**, a **95% confidence interval**, **per-domain** accuracy, and how often the two methods **agree**. We also list a few wrong answers — the seed of your failure analysis.

**Why:** these numbers are the "before" scoreboard. Per-domain tells you *where* the model is weak, and therefore where fine-tuning has the most room to help.

**How to read it:**
- Random guessing ≈ **25%** (4 options). A strong base model should be well above that.
- The **±** is the uncertainty from having only 100 questions — remember it before celebrating small gains later.
- **Method agreement** high (e.g. > 90%) means the score is stable and trustworthy.

**How we check:** it prints the scoreboard and saves `baseline_summary.json`.

In [ ]:
n = len(results)
correct = sum(r["correct"] for r in results)
acc = correct / n
se = math.sqrt(acc * (1 - acc) / n)      # standard error of a proportion
ci95 = 1.96 * se
agree = sum(r["methods_agree"] for r in results) / n

print(f"BASE MODEL BASELINE   (n = {n})")
print(f"  overall accuracy : {acc*100:5.1f}%   (95% CI +/- {ci95*100:.1f} points)")
print(f"  random guessing  : ~25%   (4 options)")
print(f"  method agreement : {agree*100:.0f}%   (log-prob vs generation)")

print("\nPer-domain accuracy:")
by_dom = defaultdict(lambda: [0, 0])     # domain -> [correct, total]
for r in results:
    by_dom[r["domain"]][0] += r["correct"]
    by_dom[r["domain"]][1] += 1
for dom, (c, t) in sorted(by_dom.items()):
    print(f"  {dom:<18} {c:>3}/{t:<3} = {c/t*100:5.1f}%")

print("\nSample wrong answers (start of your failure analysis):")
for r in [r for r in results if not r["correct"]][:8]:
    print(f"  {r['id']} [{r['domain']}]  gold={r['gold']}  model={r['pred_logprob']}")

summary = {
    "model": MODEL,
    "n": n,
    "accuracy": acc,
    "ci95": ci95,
    "method_agreement": agree,
    "per_domain": {d: {"correct": c, "total": t} for d, (c, t) in by_dom.items()},
}
with open(f"{OUT}/baseline_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("\nSaved:", f"{OUT}/baseline_summary.json")
print("This accuracy is your 'BEFORE' number — Day 4 trains the model, then we compare.")

## What you produced today
- `baseline_per_question.csv` — every question with the base model's pick (both methods).
- `baseline_summary.json` — overall accuracy, 95% CI, per-domain accuracy, method agreement.
- Your **"before" number** — the score the fine-tuned model must beat.

### Save your work on Kaggle (important!)
`/kaggle/working/` is wiped when the session ends — click **Save Version** (top-right) now, or download the files from the **Output** tab.

### Day 3 checklist
- [ ] GPU + Internet turned on
- [ ] Benchmark loaded (ideally the full 100 questions)
- [ ] Base model loaded in 4-bit (fp16 compute)
- [ ] Both scorers working (smoke test passed)
- [ ] Whole exam graded, `baseline_per_question.csv` saved
- [ ] Overall + per-domain accuracy + 95% CI recorded
- [ ] `baseline_summary.json` saved and **Version saved**

### Next — Day 4 (the fun one)
**Fine-tune** the model with QLoRA on your `train.jsonl`, watching the validation loss to avoid overfitting. Then Days 5–6 we re-score with **this same harness** and compare against today's baseline — that's how we prove whether it actually improved. Keep the GPU on for Day 4.